In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

np.random.seed(42)



In [2]:
# Load a representative subset (500k rows) to keep runtime ≤600 s while retaining data distribution
train_df = pd.read_csv(
    "../input/new-york-city-taxi-fare-prediction/train.csv", nrows=500_000
)
test_df = pd.read_csv("../input/new-york-city-taxi-fare-prediction/test.csv")



In [3]:
train_df.isnull().sum()



key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    5
dropoff_latitude     5
passenger_count      0
dtype: int64

In [4]:
train_df.dropna(axis=0, subset=["dropoff_longitude", "dropoff_latitude"], inplace=True)
train_df.reset_index(drop=True, inplace=True)



In [5]:
pd.set_option("display.float_format", lambda x: "%.5f" % x)
train_df.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,499995.00000,499995.00000,499995.00000,499995.00000,499995.00000,499995.00000
mean,11.35818,-72.52009,39.92035,-72.52244,39.91653,1.68344
std,9.91607,11.85645,8.07332,11.79736,7.39100,1.30739
min,-44.90000,-2986.24249,-3116.28538,-3383.29661,-2559.74891,0.00000
25%,6.00000,-73.99205,40.73492,-73.99138,40.73406,1.00000
50%,8.50000,-73.98179,40.75267,-73.98013,40.75315,1.00000
75%,12.50000,-73.96712,40.76708,-73.96357,40.76814,2.00000
max,500.00000,2140.60116,1703.09277,40.85103,404.61667,6.00000


In [6]:
print("Number of observations out of valid range in coordinate columns:")
print(
    "pickup_longitude:",
    (train_df.pickup_longitude < -180).sum() + (train_df.pickup_longitude > 180).sum(),
)
print(
    "pickup_latitude :",
    (train_df.pickup_latitude < -90).sum() + (train_df.pickup_latitude > 90).sum(),
)
print(
    "dropoff_longitude:",
    (train_df.dropoff_longitude < -180).sum()
    + (train_df.dropoff_longitude > 180).sum(),
)
print(
    "dropoff_latitude :",
    (train_df.dropoff_latitude < -90).sum() + (train_df.dropoff_latitude > 90).sum(),
)



Number of observations out of valid range in coordinate columns:
pickup_longitude: 8
pickup_latitude : 6
dropoff_longitude: 6
dropoff_latitude : 5


In [7]:
coord_mask = (
    (train_df.pickup_longitude < -180)
    | (train_df.pickup_longitude > 180)
    | (train_df.pickup_latitude < -90)
    | (train_df.pickup_latitude > 90)
    | (train_df.dropoff_longitude < -180)
    | (train_df.dropoff_longitude > 180)
    | (train_df.dropoff_latitude < -90)
    | (train_df.dropoff_latitude > 90)
)
train_df = train_df[~coord_mask].reset_index(drop=True)



In [8]:
train_df.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,499976.00000,499976.00000,499976.00000,499976.00000,499976.00000,499976.00000
mean,11.35816,-72.51083,39.92297,-72.50708,39.92206,1.68345
std,9.91601,10.41604,6.10202,10.42475,6.10641,1.30738
min,-44.90000,-128.17595,-74.01619,-121.39125,-74.03520,0.00000
25%,6.00000,-73.99205,40.73492,-73.99138,40.73406,1.00000
50%,8.50000,-73.98178,40.75267,-73.98013,40.75315,1.00000
75%,12.50000,-73.96712,40.76708,-73.96357,40.76813,2.00000
max,500.00000,40.81289,69.40000,40.85103,73.99726,6.00000


In [9]:
swap_idx = train_df[train_df.pickup_longitude >= 40].index
train_df.loc[swap_idx, ["dropoff_longitude", "dropoff_latitude"]] = train_df.loc[
    swap_idx, ["dropoff_latitude", "dropoff_longitude"]
].values
train_df.loc[swap_idx, ["pickup_longitude", "pickup_latitude"]] = train_df.loc[
    swap_idx, ["pickup_latitude", "pickup_longitude"]
].values



In [10]:
geo_mask = (
    (train_df.pickup_longitude < -75)
    | (train_df.pickup_longitude > -72)
    | (train_df.dropoff_longitude < -75)
    | (train_df.dropoff_longitude > -72)
    | (train_df.pickup_latitude < 40)
    | (train_df.pickup_latitude > 42)
    | (train_df.dropoff_latitude < 40)
    | (train_df.dropoff_latitude > 42)
)
train_df = train_df[~geo_mask].reset_index(drop=True)



In [11]:
train_df.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,489817.00000,489817.00000,489817.00000,489817.00000,489817.00000,489817.00000
mean,11.34232,-73.97516,40.75102,-73.97419,40.75141,1.68414
std,9.84520,0.03920,0.03008,0.03868,0.03322,1.30785
min,-44.90000,-74.96814,40.05272,-74.96426,40.04118,0.00000
25%,6.00000,-73.99227,40.73655,-73.99157,40.73561,1.00000
50%,8.50000,-73.98208,40.75339,-73.98058,40.75388,1.00000
75%,12.50000,-73.96833,40.76749,-73.96526,40.76842,2.00000
max,500.00000,-72.70287,41.80025,-72.19609,41.92382,6.00000


In [12]:
train_df.passenger_count.value_counts()



passenger_count
1    338785
2     72532
5     34622
3     21339
6     10401
4     10384
0      1754
Name: count, dtype: int64

In [13]:
train_df = train_df.drop(train_df[train_df.passenger_count == 0].index, axis=0)



In [14]:
train_df.fare_amount.sort_values(ascending=False)



99787    500.00000
242677   495.00000
229127   450.00000
281839   450.00000
322377   450.00000
            ...   
471615    -5.00000
292406    -6.50000
317123    -6.50000
418878   -18.10000
57745    -44.90000
Name: fare_amount, Length: 488063, dtype: float64

In [15]:
train_df = train_df.drop(train_df[train_df.fare_amount <= 0].index, axis=0)
train_df = train_df[train_df.fare_amount < 200]  # cap high fares
train_df["fare_amount"].sort_values(ascending=False)



156007   199.50000
419667   197.00000
272052   196.00000
250614   190.88000
313709   185.00000
            ...   
190231     0.01000
213528     0.01000
88155      0.01000
2725       0.01000
275470     0.01000
Name: fare_amount, Length: 488003, dtype: float64

In [16]:
test_df.isna().sum()



key                  0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
dtype: int64

In [17]:
test_df.describe()



,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,9914.00000,9914.00000,9914.00000,9914.00000,9914.00000
mean,-72.36544,39.85407,-72.35530,39.84990,1.70173
std,10.90123,6.32535,10.92825,6.33854,1.32833
min,-74.17807,-74.00043,-75.43773,-74.00951,0.00000
25%,-73.99227,40.73469,-73.99136,40.73366,1.00000
50%,-73.98198,40.75250,-73.98022,40.75321,1.00000
75%,-73.96672,40.76711,-73.96360,40.76814,2.00000
max,40.79030,41.36614,40.80071,41.05638,6.00000


In [18]:
train_df.dtypes



key                   object
fare_amount          float64
pickup_datetime       object
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
passenger_count        int64
dtype: object

In [19]:
train_df["pickup_datetime"] = pd.to_datetime(train_df["pickup_datetime"])
test_df["pickup_datetime"] = pd.to_datetime(test_df["pickup_datetime"])




In [20]:
def date_splitter(df):
    df["Year"] = df["pickup_datetime"].dt.year
    df["Month"] = df["pickup_datetime"].dt.month
    df["Day"] = df["pickup_datetime"].dt.day
    df["Weekday"] = df["pickup_datetime"].dt.dayofweek
    df["Hour"] = df["pickup_datetime"].dt.hour


date_splitter(train_df)
date_splitter(test_df)

train_df.drop(["pickup_datetime"], axis=1, inplace=True)
test_df.drop(["pickup_datetime"], axis=1, inplace=True)



In [21]:
import math


def haversine_distance(df):
    coord = [
        "pickup_latitude",
        "pickup_longitude",
        "dropoff_latitude",
        "dropoff_longitude",
    ]
    phi1, lambda1, phi2, lambda2 = [df[i] * math.pi / 180.0 for i in coord]
    R = 6371
    dPhi = phi2 - phi1
    dLambda = lambda2 - lambda1
    a = (
        np.sin(dPhi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(dLambda / 2.0) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    df["Distance"] = R * c

    df["Delta_Lat"] = (df["dropoff_latitude"] - df["pickup_latitude"]).abs()
    df["Delta_Lon"] = (df["dropoff_longitude"] - df["pickup_longitude"]).abs()
    df["Is_Weekend"] = (df["Weekday"] >= 5).astype(int)
    df["Dist_per_passenger"] = df["Distance"] / (df["passenger_count"] + 1)

    df["Log_Distance"] = np.log1p(df["Distance"])
    df["Is_Night"] = ((df["Hour"] >= 0) & (df["Hour"] <= 5)).astype(int)

    df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
    df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)


haversine_distance(train_df)
haversine_distance(test_df)



In [22]:
train_df.Distance.sort_values()



441728     0.00000
62568      0.00000
352567     0.00000
62588      0.00000
62611      0.00000
            ...   
215588   102.14249
57047    103.40547
400457   103.69855
28420    104.03496
217678   110.83308
Name: Distance, Length: 488003, dtype: float64

In [23]:
pass



In [24]:
pass



In [25]:
pass



In [26]:
pass



In [27]:
pass



In [28]:
pass



In [29]:
pass



In [30]:
features = [col for col in train_df.columns if col not in ["key", "fare_amount"]]
target = "fare_amount"



In [31]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor



In [32]:
# Convert feature frames to float32 once to speed up training and reduce memory traffic
train_X = train_df[features].astype(np.float32).to_numpy(copy=False)
val_X = (
    train_df[features].astype(np.float32).to_numpy(copy=False)
)  # placeholder; will be split below
test_X = test_df[features].astype(np.float32).to_numpy(copy=False)

X_train, X_val, y_train, y_val = train_test_split(
    train_X, train_df[target].values, test_size=0.30, random_state=42
)



In [33]:
y_train_log = np.log1p(y_train).astype(np.float32)
y_val_log = np.log1p(y_val).astype(np.float32)

log_model = XGBRegressor(
    n_estimators=8000,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    n_jobs=os.cpu_count(),
    random_state=42,
    reg_lambda=1.0,
    tree_method="hist",
    predictor="cpu_predictor",
    max_bin=256,
    single_precision_histogram=True,
)

log_model.fit(
    X_train,
    y_train_log,
    eval_set=[(X_val, y_val_log)],
    eval_metric="rmse",
    early_stopping_rounds=800,
    verbose=False,
)

val_pred_log = log_model.predict(X_val)
val_pred_log_back = np.expm1(val_pred_log)
val_rmse_log = mean_squared_error(y_val, val_pred_log_back, squared=False)

raw_model = XGBRegressor(
    n_estimators=6000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    n_jobs=os.cpu_count(),
    random_state=42,
    reg_lambda=2.0,
    reg_alpha=0.0,
    tree_method="hist",
    predictor="cpu_predictor",
    max_bin=256,
    single_precision_histogram=True,
)

raw_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    early_stopping_rounds=300,
    verbose=False,
)

val_pred_raw = raw_model.predict(X_val)
val_rmse_raw = mean_squared_error(y_val, val_pred_raw, squared=False)

print(f"Validation RMSE (log‑model): {val_rmse_log:.5f}")
print(f"Validation RMSE (raw‑model): {val_rmse_raw:.5f}")

if val_rmse_raw < val_rmse_log:
    best_model = raw_model
    test_pred = best_model.predict(test_X)
else:
    best_model = log_model
    test_pred_log = best_model.predict(test_X)
    test_pred = np.expm1(test_pred_log)

print(f"Selected model validation RMSE: {min(val_rmse_log, val_rmse_raw):.5f}")



/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [06:34:17] WARNING: /workspace/src/learner.cc:742: 
Parameters: { "predictor", "single_precision_histogram" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dis

Validation RMSE (log‑model): 3.74650
Validation RMSE (raw‑model): 3.67770
Selected model validation RMSE: 3.67770


In [34]:
submission = pd.DataFrame({"key": test_df["key"], "fare_amount": test_pred})
submission.to_csv("taxi_fare_submission.csv", index=False)